# Tobacco Data Gateway — Example Queries

This notebook demonstrates how to use the `tobacco_gateway` package to fetch and explore tobacco-related data for Munich → Bavaria → Germany.

Run from the repo root with the venv active: `.venv/bin/jupyter notebook`

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path when running from notebooks/

from tobacco_gateway import fetch, query
import pandas as pd

## 1. Which sources can answer a question?

Use `query()` to find relevant sources before fetching data.

In [ ]:
results = query("e-cigarette use trend by age group")
for r in results:
    print(f"{r['score']:4.1f}  {r['id']:35s}  {r['geographic_level']}")

In [ ]:
# Geographic cascade: prefer Munich, fall back to Bavaria, then Germany
results = query("smoking prevalence Munich")
for r in results:
    print(f"{r['score']:4.1f}  {r['id']:35s}  {r['geographic_level']}")

## 2. Germany-level smoking trend (Destatis Mikrozensus)

The Mikrozensus covers 2009–2017 with district-level detail. Freely available, auto-downloaded.

In [ ]:
mz = fetch("destatis_mikrozensus")
print(f"Shape: {mz.shape}")
mz.head()

In [ ]:
# Smoking prevalence over time (total population, both sexes)
total = mz[mz['sex'] == 'insgesamt'].copy()
total['smoker_pct'] = total['smokers_total_1000'] / total['pop_with_data_1000'] * 100
total[['year', 'smoker_pct']].dropna().sort_values('year')

## 3. Bavaria-level reports (BGS Bayern)

LGL Bayern publishes health and addiction monitoring reports. PDFs are downloaded to `data/bgs_bayern/`.

In [ ]:
bgs = fetch("bgs_bayern")
bgs

The 2021 Suchtmonitoring Bayern report focuses specifically on smoking behavior.
Open the PDF to extract specific tables using `pdfplumber`:

In [ ]:
import pdfplumber

pdf_path = bgs.loc[bgs['year'] == 2021, 'pdf_path'].iloc[0]
with pdfplumber.open(pdf_path) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    # Search for pages mentioning Rauchen
    for i, page in enumerate(pdf.pages[:5]):
        text = page.extract_text() or ""
        if 'Rauchen' in text or 'rauchen' in text:
            print(f"  Page {i+1}: contains smoking content")

## 4. Munich-level reports

The Münchner Gesundheitsbericht and Gesundheitsbefragung are the most local sources.

In [ ]:
gb = fetch("muenchen_gesundheitsbericht")
gbf = fetch("muenchen_gesundheitsbefragung")
pd.concat([gb, gbf], ignore_index=True)

## 5. Sources requiring manual steps

Some sources require a data-use agreement or manual export. `fetch()` raises a `RuntimeError` with step-by-step instructions.

In [ ]:
for src in ['rki_geda', 'staba_mikrozensus', 'rki_kiggs', 'eurobarometer_tobacco']:
    try:
        fetch(src)
        print(f"{src}: OK")
    except RuntimeError as e:
        print(f"{src}: manual step needed — {str(e).splitlines()[0]}")

## 6. All sources at a glance

In [ ]:
import pathlib, re

rows = []
for source_dir in sorted(pathlib.Path('../sources').iterdir()):
    md = source_dir / 'SOURCE.md'
    if not md.exists():
        continue
    text = md.read_text()
    # Extract key YAML fields
    def field(key):
        m = re.search(rf'^{key}: (.+)$', text, re.MULTILINE)
        return m.group(1).strip('"') if m else ''
    rows.append({
        'id': field('id'),
        'level': field('geographic_level'),
        'years': field('years_available'),
        'access': field('access_method'),
        'format': field('data_format'),
    })

pd.DataFrame(rows)